## Basic video processing flow

→ open video/camera
→ read frame
→ process frame
→ display/write frame
→ repeat
→ release resources

# Read a video file frame by frame

In [3]:
import cv2

# helpers
def _convert_from_bgr_to_rgb(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

def _convert_from_bgr_to_gray(img):
    return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)



In [4]:
from IPython.display import display, Image
import ipywidgets as widgets
import io

out = widgets.Output()
display(out)

video_path = "videos/person_thermal.mkv"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    print("Could not open video")

while True:

    # ret == True   → frame was read correctly
    # ret == False  → no frame available / end of file / error
    ret, frame = cap.read()

    if not ret:
        print("End of video or can not read frame")
        break

    rgb = _convert_from_bgr_to_rgb(frame)
    _, buf = cv2.imencode(".jpg", rgb)

    with out:
        out.clear_output(wait=True)
        display(Image(data=buf.tobytes()))

cap.release()

Output()

End of video or can not read frame


## Inspect video properties

In [5]:
cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
width = cap.get(cv2.CAP_PROP_FRAME_WIDTH)
height = cap.get(cv2.CAP_PROP_FRAME_HEIGHT)
frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)

print("FPS:", fps)
print("Width:", width)
print("Height:", height)
print("Frame count:", frame_count)

# always important
cap.release()

FPS: 60.0
Width: 640.0
Height: 480.0
Frame count: 1161.0


## Display video at approximately correct speed
This code is not working in the notebook but code stays here for reference

In [ ]:
cap = cv2.VideoCapture("videos/input.mp4")

fps = cap.get(cv2.CAP_PROP_FPS)
delay = int(1000 / fps) if fps > 0 else 30

while True:
    ret, frame = cap.read()
    if not ret:
        break

    cv2.imshow("Video normal speed", frame)

    if cv2.waitKey(delay) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

# Read from a webcam or camera
This code is not working in the notebook but code stays here for reference

In [ ]:
import cv2

cap = cv2.VideoCapture(0)

if not cap.isOpened():
    raise RuntimeError("Could not open camera")

while True:
    ret, frame = cap.read()

    if not ret:
        print("Could not read frame")
        break

    cv2.imshow("Camera", frame)

    if cv2.waitKey(1) & 0xFF == ord("q"):
        break

cap.release()
cv2.destroyAllWindows()

# Apply image processing inside the frame loop

In [8]:

save_path = "outputs/videos/"
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Could not open video")

# Some parameters needed by the video writter
fps = cap.get(cv2.CAP_PROP_FPS)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

if fps < 30:
    fps = 30

# Selecting video codec with the fourcc standard
fourcc = cv2.VideoWriter_fourcc(*"mp4v")

writer = cv2.VideoWriter(
    save_path + "edges_video.mp4",
    fourcc,
    fps,
    (width, height),
    isColor=False
)

while True:

    # read frames
    ret, frame = cap.read()
    if not ret:
        break

    # frame processing
    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    edges = cv2.Canny(blurred, 100, 200)

    # save each frame processed
    writer.write(edges)

cap.release()
writer.release()
print("Video saved")

Video saved


# Measure actual processing FPS

In [9]:
import time

cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Video could not be opened")

fps = cap.get(cv2.CAP_PROP_FPS)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

if fps < 30:
    fps = 30

print("fps: ", fps)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(save_path + "fps_processing_time.mp4", fourcc, fps, (width, height))

#time before first processing
prev_time = time.time() 
while True:

    ret, frame = cap.read()
    if not ret:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    edges = cv2.Canny(blurred, 100, 200)
    color = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    # counting processing time
    now = time.time()
    processing_fps = 1.0 / (now - prev_time)
    prev_time = now
    
    # tag each frame with the processing fps
    cv2.putText(
        color,
        f"FPS: {processing_fps:1f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (255, 0, 0),
        2
    )

    writer.write(color)

cap.release()
writer.release()
print("Video saved")

fps:  60.0
Video saved


# Smooth FPS

In [11]:
cap = cv2.VideoCapture(video_path)

if not cap.isOpened():
    raise RuntimeError("Video could not be opened")

fps = cap.get(cv2.CAP_PROP_FPS)
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))

if fps < 30:
    fps = 30

print("fps: ", fps)
fourcc = cv2.VideoWriter_fourcc(*"mp4v")
writer = cv2.VideoWriter(save_path + "fps_smooth_processing_time.mp4", fourcc, fps, (width, height))

#time before first processing
prev_time = time.time()
smooth_fps = 0
while True:

    ret, frame = cap.read()
    if not ret:
        break

    gray_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    blurred = cv2.GaussianBlur(gray_frame, (5, 5), 0)
    edges = cv2.Canny(blurred, 100, 200)
    color = cv2.cvtColor(edges, cv2.COLOR_GRAY2BGR)

    # counting processing time
    now = time.time()
    processing_fps = 1.0 / (now - prev_time)
    smooth_fps = 0.99 * smooth_fps + 0.01 * processing_fps if smooth_fps > 0 else processing_fps # the initial weigths were 0.9 and 0.1
    prev_time = now
    
    # tag each frame with the processing fps
    cv2.putText(
        color,
        f"FPS: {smooth_fps:1f}",
        (20, 40),
        cv2.FONT_HERSHEY_SIMPLEX,
        1.0,
        (255, 0, 0),
        2
    )

    writer.write(color)

cap.release()
writer.release()
print("Video saved")

fps:  60.0
Video saved


## Basic Latency Idea

time between frame capture and frame display/output

With `VideoCapture`, exact camera latency is not always easy to measure, but you can reduce practical latency by:
* using smaller resolution
* using lower processing cost
* avoiding unnecessary copies
* keeping buffers small
* dropping old frames
* using GStreamer pipelines later

For live cameras, a common issue is: your program processes old frames while new frames wait in a buffer.
For real-time systems, it is often better to drop old frames and process the newest one.



# Video processing is image processing inside a loop.